# Adversarial TTS - Class-Based Architecture

This notebook uses the refactored class-based architecture:
- **EnvironmentLoader**: Handles model loading and environment setup
- **AdversarialTrainer**: Runs the optimization loop (returns results)
- **RunLogger**: Handles all output and logging (called separately)

This separation allows you to re-run logging/visualization without re-running optimization.

## 1. Configuration

In [1]:
import torch

# Import class-based modules

class NotebookArgs:
    """Configuration class mimicking argparse for notebook usage."""
    def __init__(self):
        # String parameters
        self.ground_truth_text = "I think the NFL is lame and boring"
        self.target_text = "The Seattle Seahawks are the best Team in the world"

        # Numeric parameters
        self.loop_count = 1
        self.num_generations = 4
        self.pop_size = 4
        self.iv_scalar = 0.5
        self.size_per_phoneme = 1
        self.batch_size = -1

        # Boolean parameters
        self.notify = False
        self.subspace_optimization = False
        self.multi_gpu = False  # Set to True to enable multi-GPU

        # Enum/Selection parameters
        self.mode = "TARGETED"
        self.ACTIVE_OBJECTIVES = ["PESQ", "WHISPER_PROB"]
        self.thresholds = ["PESQ=0.2", "WHISPER_PROB=0.6"]


args = NotebookArgs()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")

Device: cpu
Available GPUs: 0


## 2. Initialize Environment

In [2]:
from Trainer.ModelLoader import EnvironmentLoader

# Initialize environment using EnvironmentLoader
loader = EnvironmentLoader(args, device)
config, models, audio, embeds, objective_manager = loader.initialize()

if config is None:
    raise RuntimeError("Initialization failed.")

print("\nEnvironment initialized successfully!")

=== Configuration ===
Mode:              TARGETED
GT Text:           I think the NFL is lame and boring
Target Text:       The Seattle Seahawks are the best Team in the world
Generations:       4
Population Size:   4
Loop Count:        1
IV Scalar:         0.5
Size Per Phoneme:  1
Notify (WhatsApp): False
Objectives:        ['PESQ', 'WHISPER_PROB']
Thresholds:        PESQ<=0.2, WHISPER_PROB<=0.6
Multi-GPU:         Disabled
Loading StyleTTS2...
Loading ASR Model (Whisper)...

[INFO] Initializing ObjectiveManager...
[ObjectiveManager] Initializing 2 objectives...
  [OK] PESQ (batching=False)
  [OK] WHISPER_PROB (batching=True)
[ObjectiveManager] All objectives initialized.

Environment initialized successfully!


## 3. Run Optimization

The trainer only runs optimization and returns results. Logging is done separately.

In [3]:
from Trainer.AdversarialTrainer import AdversarialTrainer

# Create trainer (optimization only - no logging)
trainer = AdversarialTrainer(
    config=config,
    models=models,
    audio=audio,
    embeds=embeds,
    objective_manager=objective_manager,
    device=device
)

# Run optimization - returns list of OptimizationResult
results = trainer.run()

print(f"\nOptimization complete! Got {len(results)} result(s).")

[Info] Starting Training for 1 loops...

--- Optimization Loop 1/1 ---


Generation Loop 1:   0%|          | 0/4 [00:00<?, ?it/s]


Optimization complete! Got 1 result(s).


## 4. Log Results (Separate Step)

You can re-run this cell with different logging settings without re-running optimization.

In [5]:
import importlib
import sys

# Find the module in the system's loaded modules
runloggermodule = 'Trainer.RunLogger'
graphplottermodule = 'Trainer.GraphPlotter'
if runloggermodule in sys.modules:
    importlib.reload(sys.modules[runloggermodule])
    print(f"Successfully reloaded {runloggermodule}")

if graphplottermodule in sys.modules:
    importlib.reload(sys.modules[graphplottermodule])
    print(f"Successfully reloaded {graphplottermodule}")

# Now import the class from the freshly reloaded module
from Trainer.RunLogger import RunLogger

# Create logger and save results
logger = RunLogger(config, models, audio, device)

for i, result in enumerate(results):
    print(f"Saving results for loop {i + 1}...")
    logger.finalize_run(
        result.fitness_data,
        result.generation_count,
        result.elapsed_time
    )

print(f"\nResults saved to: {logger.folder_path}")

Successfully reloaded Trainer.RunLogger
Successfully reloaded Trainer.GraphPlotter
[Info] Output directory initialized: outputs/PESQ_WHISPER_PROB/20260108_2207
Saving results for loop 1...
[Log] Pareto evolution graph saved to outputs/PESQ_WHISPER_PROB/20260108_2207/pareto_evolution.png
[Log] Mean fitness graph saved to outputs/PESQ_WHISPER_PROB/20260108_2207/mean_fitness_stack.png
[Log] Minimal fitness graph saved to outputs/PESQ_WHISPER_PROB/20260108_2207/minimal_fitness_stack.png


/home/yanis/miniconda3/envs/styletts2/lib/python3.9/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


[Log] Torch state saved for reproducibility: outputs/PESQ_WHISPER_PROB/20260108_2207/reconstruction_pack.pt

Results saved to: outputs/PESQ_WHISPER_PROB/20260108_2207


## 5. Re-generate Visualizations (Optional)

You can regenerate graphs with different settings without re-running optimization.

In [6]:
# Example: Regenerate visualizations with the existing results
from Trainer.GraphPlotter import GraphPlotter

# Use the last result's fitness data
if results:
    last_result = results[-1]
    
    # Create a new plotter (you could customize settings here)
    plotter = GraphPlotter(
        folder_path=logger.folder_path,
        active_objectives=config.active_objectives,
        num_generations=config.num_generations
    )
    
    # Regenerate all visualizations
    plotter.generate_all_visualizations(last_result.fitness_data)
    print("Visualizations regenerated!")

TypeError: __init__() got an unexpected keyword argument 'num_generations'